In [2]:
"""Load the energy weather residual load in Postgres."""

import datetime as dt
import polars as pl
from uuid import UUID
from zoneinfo import ZoneInfo

from impuls_algo.interfaces.settings import EnappsysSettings
from impuls_algo.interfaces.postgres import PostgresDbSettings, PostgresInterface
from impuls_algo.models.data import Metric, PostgresDataId, Scope
from impuls_interfaces.aws import BaseSsmParameters
from pydantic import ConfigDict, Json



class SharedSettings(BaseSsmParameters):
    """Shared Settings."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    __ssm_namespace__ = "/algo/shared"

    db: Json[PostgresDbSettings]
    enappsys: Json[EnappsysSettings]


settings = SharedSettings.from_ssm()


delivery_period_start = dt.datetime(2026, 5, 1)
delivery_period_end = dt.datetime(2026, 5, 10)
# enery weather residual load forecast data
residual_load_postgres_data_id = PostgresDataId(
    metric=Metric(id=UUID("d5600a9e-ac52-4008-87c3-b4651378c7d8"), name="residual_load_fcs"),
    scope=Scope(id=UUID("89d1f2da-771c-42b5-b24a-79c9baf60c68"), name="de"),
    table="forecast_versioned",
    db_schema="market_data",
)



postgres_interface = PostgresInterface(
    db_settings=settings.db,
    local_zoneinfo=ZoneInfo("Europe/Berlin"),
)
# load data



residual_load_results = postgres_interface.load_data(
    data_ids=[residual_load_postgres_data_id],
    delivery_period_start=delivery_period_start,
    delivery_period_end=delivery_period_end,
   
)

print(residual_load_results)

2026-09-03 16:41:44.087 | DEBUG    | impuls_interfaces.db.postgres.connection:create_iam_db_connection:110 - Using IAM authentication for database.
2026-09-03 16:41:44.266 | DEBUG    | impuls_interfaces.db.postgres.connection:create_iam_db_connection:141 - Registering application name for PostgreSQL connection as 'impuls_algo'
2026-09-03 16:41:44.347 | INFO     | impuls_algo.interfaces.postgres:load_data:93 - Load residual_load_fcs for de from postgres.


shape: (1_003_943, 4)
┌────────────────────────┬─────────────────────┬───────────────────────────────┬───────────────────┐
│ inserted_at            ┆ issued              ┆ delivery_start                ┆ residual_load_fcs │
│ ---                    ┆ ---                 ┆ ---                           ┆ ---               │
│ datetime[μs,           ┆ datetime[μs,        ┆ datetime[μs, Europe/Berlin]   ┆ f64               │
│ Europe/Berlin]         ┆ Europe/Berlin]      ┆                               ┆                   │
╞════════════════════════╪═════════════════════╪═══════════════════════════════╪═══════════════════╡
│ 2026-04-06             ┆ 2026-04-06 02:00:00 ┆ 2026-05-01 02:00:00 CEST      ┆ 26955.859509      │
│ 02:32:31.406547 CES…   ┆ CEST                ┆                               ┆                   │
│ 2026-04-06             ┆ 2026-04-06 03:00:00 ┆ 2026-05-01 02:00:00 CEST      ┆ 26955.859509      │
│ 03:02:31.776028 CES…   ┆ CEST                ┆                     

In [3]:
residual_load_results.write_parquet("residual_load_results.parquet5")